#Prerequisite

##Load data

In [0]:
df=spark.read.csv(path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv", header=True, inferSchema=True)
display(df)

#Transactions

## Transaction 00- Write dataframe as Delta Format


In [0]:
df.write.save(format="DELTA", mode="overwrite", path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/output/output_delta")

## Transaction 01- Overwrite existing Delta Folder

In [0]:
from pyspark.sql.functions import col

df.filter(col("country") == "India").write.save(
    format="DELTA",
    mode="overwrite",
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/output/output_delta",
)

##Read Delta

In [0]:
spark.read.load(
    format="DELTA",
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/output/output_delta",
).display()

##Advantage- Maintains versions

In [0]:
spark.read.option("versionAsOf", 0).load(
    format="DELTA",
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/output/output_delta",
).display()

#Read transactions log

##Approach 01

In [0]:
spark.read.load(format="text", path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/output/output_delta/_delta_log/00000000000000000001.json").display()

##Approach 02 - Using Delta Table API

In [0]:
from delta.tables import DeltaTable

delta_table= DeltaTable.forPath(spark, "/Volumes/quickstart_catalog/quickstart_schema/sandbox/output/output_delta/")

delta_table.history().display()


#Advantage - Updates are allowed

- Create a DataFrame from delta source
- Create a view on top of delta
- Execute update QUERY

In [0]:
spark.read.load(
    format="DELTA",
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/output/output_delta",
).createOrReplaceTempView(
    "users_vw"
)


In [0]:
%sql
UPDATE 
    users_vw
SET 
    country = 'Bharat'
WHERE 
    country = 'India';

In [0]:
spark.read.load(
    format="DELTA",
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/output/output_delta",
).display()

In [0]:
from delta.tables import DeltaTable

delta_table= DeltaTable.forPath(spark, "/Volumes/quickstart_catalog/quickstart_schema/sandbox/output/output_delta/")

delta_table.history().display()


#Advantage- Data Quality

In [0]:
spark.read.csv(
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_006_new_column_education.csv",
    header=True,
    inferSchema=True,
).write.save(
    format="DELTA",
    mode="overwrite",
    path="/Volumes/quickstart_catalog/quickstart_schema/sandbox/output/output_delta",
)
 
spark.read.parquet(
    "/Volumes/quickstart_catalog/quickstart_schema/sandbox/output/output_delta"
).display()